# Aethon Dynamics — Knowledge Base Ingest

**Pipeline:**  
`Load .md files` → `LLM chunking (GPT-4o-mini)` → `Embed (text-embedding-3-small)` → `Store in ChromaDB`

Run cells top to bottom. ChromaDB will be persisted at `./chroma_db` (created automatically).

## 0 — Install dependencies

In [1]:
%pip install -q chromadb openai python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1 — Config & API client

In [2]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path=os.path.join("..", ".env"))

from openai import OpenAI
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

KB_DIR          = "./Knowledge-Base"
CHROMA_PATH     = "./chroma_db"
COLLECTION_NAME = "aethon_kb"
LLM_MODEL       = "gpt-4o-mini"
EMBED_MODEL     = "text-embedding-3-small"
CACHE_FILE      = "./chunks_cache.json"   # chunks + embeddings saved here

print("Config OK")

Config OK


## 2 — Load raw documents

In [3]:
import pathlib

def load_documents(kb_dir):
    docs = []
    for path in sorted(pathlib.Path(kb_dir).glob("*.md")):
        text = path.read_text(encoding="utf-8")
        docs.append({"filename": path.name, "content": text})
        print(f"  Loaded: {path.name}  ({len(text):,} chars)")
    return docs

docs = load_documents(KB_DIR)
print(f"\nTotal documents loaded: {len(docs)}")

  Loaded: achievements.md  (2,231 chars)
  Loaded: company_overview.md  (1,783 chars)
  Loaded: employees.md  (2,384 chars)
  Loaded: products.md  (2,331 chars)
  Loaded: sales.md  (1,947 chars)

Total documents loaded: 5


## 3 — LLM chunking with Structured Outputs

In [4]:
import json, time

CHUNK_SCHEMA = {
    "type": "json_schema",
    "json_schema": {
        "name": "chunked_document",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "chunks": {
                    "type": "array",
                    "description": "Ordered list of semantic chunks from the document.",
                    "items": {
                        "type": "object",
                        "properties": {
                            "text":  {"type": "string", "description": "Verbatim chunk content. Do NOT summarise or paraphrase."},
                            "topic": {"type": "string", "description": "One short phrase describing what this chunk is about."}
                        },
                        "required": ["text", "topic"],
                        "additionalProperties": False
                    }
                }
            },
            "required": ["chunks"],
            "additionalProperties": False
        }
    }
}

CHUNK_SYSTEM = """You are a document chunker for a retrieval system.
Split the document into self-contained semantic chunks.
Rules:
- Each chunk must make sense on its own, without needing surrounding chunks.
- Each chunk should cover ONE topic or entity.
- Preserve the original wording exactly — do NOT summarise or paraphrase.
- Keep section headings attached to their first chunk of content.
- Aim for 150-400 words per chunk."""

def llm_chunk_document(filename, content):
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": CHUNK_SYSTEM},
            {"role": "user",   "content": f"Document: {filename}\n\n{content}"},
        ],
        temperature=0,
        max_tokens=4096,
        response_format=CHUNK_SCHEMA,
    )
    parsed = json.loads(response.choices[0].message.content)
    return [c for c in parsed["chunks"] if c["text"].strip()]

all_chunks = []
for doc in docs:
    print(f"Chunking: {doc['filename']} ...", end=" ")
    chunks = llm_chunk_document(doc["filename"], doc["content"])
    for i, chunk in enumerate(chunks):
        all_chunks.append({
            "chunk_id":    f"{doc['filename'].replace('.md', '')}_chunk_{i}",
            "filename":    doc["filename"],
            "chunk_index": i,
            "topic":       chunk["topic"],
            "text":        chunk["text"],
        })
    print(f"{len(chunks)} chunks")
    time.sleep(0.3)

print(f"\nTotal chunks: {len(all_chunks)}")

Chunking: achievements.md ... 6 chunks
Chunking: company_overview.md ... 5 chunks
Chunking: employees.md ... 9 chunks
Chunking: products.md ... 5 chunks
Chunking: sales.md ... 6 chunks

Total chunks: 31


## 4 — Embed chunks

Embeddings are **saved to `chunks_cache.json` immediately** after the API call returns.  
If the kernel crashes, just restart and run **Cell 4b** to reload from cache — no API call needed.

In [5]:
# ── Cell 4a: Call the API and save to disk immediately ─────────────────────────

def embed_batch(texts, model=EMBED_MODEL):
    response = client.embeddings.create(input=texts, model=model)
    return [item.embedding for item in response.data]

print(f"Embedding {len(all_chunks)} chunks...", end=" ")
texts = [c["text"] for c in all_chunks]
embeddings = embed_batch(texts)   # single call — 35 chunks is well within limits
print(f"done  ({len(embeddings)} vectors, dim={len(embeddings[0])})")

# Save chunks + embeddings together so a kernel crash loses nothing
cache = [
    {**chunk, "embedding": emb}
    for chunk, emb in zip(all_chunks, embeddings)
]
with open(CACHE_FILE, "w", encoding="utf-8") as f:
    json.dump(cache, f)

print(f"Saved to {CACHE_FILE} — kernel can now restart safely")

Embedding 31 chunks... done  (31 vectors, dim=1536)
Saved to ./chunks_cache.json — kernel can now restart safely


In [6]:
# ── Cell 4b: Reload from cache after a kernel restart ─────────────────────────
# Run this instead of 4a if the kernel crashed and you want to skip the API call.

import json
with open(CACHE_FILE, encoding="utf-8") as f:
    cache = json.load(f)

all_chunks = [{k: v for k, v in c.items() if k != "embedding"} for c in cache]
embeddings = [c["embedding"] for c in cache]

print(f"Loaded from cache: {len(all_chunks)} chunks, dim={len(embeddings[0])}")

Loaded from cache: 31 chunks, dim=1536
